# 00 · Protocolo experimental pre-registrado

Este notebook carga `configs/prereg.yaml`, comprueba que todos los elementos de la tabla de
pre-registro estén completos, calcula su huella SHA-256 y exporta la tabla que se reproduce en el
Jupyter Book. **Congelamiento:**

```bash
git add configs/prereg.yaml && git commit -m "Pre-registro del protocolo" && git tag prereg
```

Cualquier cambio posterior se agrega a `desviaciones` en el YAML y se declara en el reporte.
Para verificar que el archivo no cambió desde la etiqueta, se usa
`git diff prereg -- configs/prereg.yaml`.

In [1]:
import hashlib
import subprocess
from pathlib import Path

import pandas as pd
import yaml

from project.config import CFG

ROOT = Path("..").resolve()
P = ROOT / "configs/prereg.yaml"
PREREG = yaml.safe_load(P.read_text(encoding="utf-8"))
print("SHA-256 del pre-registro:", hashlib.sha256(P.read_bytes()).hexdigest())
try:
    tag = subprocess.run(["git", "rev-list", "-n", "1", "prereg"], capture_output=True, text=True, cwd=ROOT)
    print("Commit de la etiqueta `prereg`:", tag.stdout.strip() or "(aún no creada: congelar antes de entrenar)")
except FileNotFoundError:
    print("git no disponible")

SHA-256 del pre-registro: d7a31addfecd2b033e72577d68b592e62a0b38156a450626db9c91e93865b4fc
Commit de la etiqueta `prereg`: (aún no creada: congelar antes de entrenar)


## Verificación de completitud

Elementos exigidos por la tabla de pre-registro: hipótesis con criterio numérico de refutación, métrica
primaria, secundarias, unidad de análisis, particiones, pruebas, δ, semillas, presupuesto y criterios de
parada.

In [2]:
REQUIRED = ["hipotesis", "metrica_primaria", "metricas_secundarias", "unidad_de_analisis", "particiones",
            "pruebas_estadisticas", "efecto_minimo_relevante", "semillas", "presupuesto", "criterios_de_parada"]
missing = [k for k in REQUIRED if not PREREG.get(k)]
for h, d in PREREG["hipotesis"].items():
    for f in ("enunciado", "contraste", "prueba", "refutacion"):
        if not d.get(f):
            missing.append(f"{h}.{f}")
assert len(PREREG["semillas"]["modelos_ligeros"]) >= 5 and len(PREREG["semillas"]["modelos_pesados"]) >= 3
assert PREREG["pruebas_estadisticas"]["alpha"] == CFG.alpha
print("Elementos faltantes:", missing or "ninguno")

Elementos faltantes: ninguno


In [3]:
tabla = pd.DataFrame([
    ("Hipótesis", "; ".join(f"{k}: {v['refutacion'].strip()}" for k, v in PREREG["hipotesis"].items())),
    ("Métrica primaria", f"Imágenes: {PREREG['metrica_primaria']['imagenes']}. NLP: {PREREG['metrica_primaria']['nlp']}"),
    ("Métricas secundarias", "Imágenes: " + ", ".join(PREREG["metricas_secundarias"]["imagenes"]) + ". NLP: " + ", ".join(PREREG["metricas_secundarias"]["nlp"])),
    ("Unidad de análisis", f"{PREREG['unidad_de_analisis']['imagenes']} / {PREREG['unidad_de_analisis']['nlp']}"),
    ("Diseño de particiones", PREREG["particiones"]["segmentacion"].strip() + " | " + PREREG["particiones"]["nlp"].strip()),
    ("Pruebas estadísticas", f"α = {PREREG['pruebas_estadisticas']['alpha']}, corrección {PREREG['pruebas_estadisticas']['correccion']}; "
     + PREREG["pruebas_estadisticas"]["tres_o_mas_modelos"]),
    ("Efecto mínimo relevante (δ)", f"AP: {PREREG['efecto_minimo_relevante']['imagenes_AP']}; F1: {PREREG['efecto_minimo_relevante']['nlp_F1']}; ECE: {PREREG['efecto_minimo_relevante']['ECE']}"),
    ("Semillas", f"ligeros {PREREG['semillas']['modelos_ligeros']}, pesados {PREREG['semillas']['modelos_pesados']}"),
    ("Presupuesto", f"{PREREG['presupuesto']['n_trials_hpo']} trials HPO por modelo; horas-GPU {PREREG['presupuesto']['horas_gpu_por_modelo']}"),
    ("Criterios de parada", "; ".join(f"{k}: {v}" for k, v in PREREG["criterios_de_parada"].items())),
], columns=["Elemento", "Declaración"])
pd.set_option("display.max_colwidth", 200)
(ROOT / "reports/tables").mkdir(parents=True, exist_ok=True)
tabla.to_csv(ROOT / "reports/tables/prereg_tabla.csv", index=False)
tabla

,Elemento,Declaración
0,Hipótesis,"H1: Se REFUTA si el IC 95 % de Δ25 incluye 0, o si el IC 95 % de (Δ25 − Δ100) incluye 0 o es negativo (la brecha no se reduce).; H2: Se REFUTA si ρ(rank_AUC, rank_AP) > 0.9 o si alguna media de AU..."
1,Métrica primaria,"Imágenes: AP@[0.5:0.95] estilo Sartorius (media sobre umbrales de IoU de TP/(TP+FP+FN), por imagen). NLP: F1 macro (salvo que el EDA justifique otra; cualquier cambio se registra como desviación)"
2,Métricas secundarias,"Imágenes: AP50, AP75, PQ, SQ, RQ, AJI, precision_instancia, recall_instancia, F1_instancia, MAE_conteo, RMSE_conteo, pearson_conteo, bland_altman, Dice, IoU, precision_pixel, recall_pixel, HD95, F..."
3,Unidad de análisis,"imagen (todas las métricas se calculan por imagen y se agregan con IC bootstrap) / ejemplo (McNemar, bootstrap en test) y par semilla×fold (t corregida)"
4,Diseño de particiones,70/15/15 (train/val/test) por grupo sample_id con StratifiedGroupKFold estratificado por tipo celular (semilla 42); test sellado evaluado una sola vez por modelo final; 5-fold agrupado de la mejor...
5,Pruebas estadísticas,"α = 0.05, corrección holm; Friedman + Wilcoxon por pares con Holm + diagrama de diferencia crítica (Nemenyi)"
6,Efecto mínimo relevante (δ),AP: 0.02; F1: 0.02; ECE: 0.01
7,Semillas,"ligeros [0, 1, 2, 3, 4], pesados [0, 1, 2]"
8,Presupuesto,"30 trials HPO por modelo; horas-GPU {'segmentacion_pesados': 6, 'nlp_transformer': 2}"
9,Criterios de parada,"segmentacion: early stopping sobre pérdida de validación, paciencia 10 épocas, máximo 100; nlp_keras: early stopping sobre pérdida de validación, paciencia 3 épocas, máximo 20/50 (rejilla); transf..."


## Registro de desviaciones

Toda desviación respecto de este protocolo se añade a `desviaciones` con fecha, sección, motivo e
impacto esperado. En el reporte se distingue entre análisis **confirmatorios** (pre-registrados) y
**exploratorios**.

In [4]:
pd.DataFrame(PREREG.get("desviaciones") or [], columns=["fecha", "seccion", "descripcion", "motivo", "impacto_esperado"])

,fecha,seccion,descripcion,motivo,impacto_esperado
